In [1]:
import requests_cache
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy  as np
# import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
import time

In [2]:
session = requests_cache.CachedSession(cache_name='cache_data', use_cache_dir=True, expire_after=604800)
standings_urls = {
    # 'PremierLeague' : "https://fbref.com/en/comps/9/Premier-League-Stats", 
    # 'Laliga' : "https://fbref.com/en/comps/12/La-Liga-Stats'",
    # 'Bundesliga' : "https://fbref.com/en/comps/20/Bundesliga-Stats", 
    # 'SerieA' : "https://fbref.com/en/comps/11/Serie-A-Stats",
    # 'League1' : "https://fbref.com/en/comps/13/ligue-1-Stats",
    # 'BelgianProLeague' : 'https://fbref.com/en/comps/37/Belgian-Pro-League-Stats',
    # 'Eredivisie' : 'https://fbref.com/en/comps/23/Eredivisie-Stats',
    'PrimeiraLiga' : 'https://fbref.com/en/comps/32/Primeira-Liga-Stats',
    'PremierLeagueWomen' : 'https://fbref.com/en/comps/189/Womens-Super-League-Stats'}


In [3]:
def get_team_links(b_data, L_soup):
    standings_table = L_soup.select('table.stats_table')[0]
    td_tags = standings_table.find_all('td', {"data-stat": 'team'})

    all_team_names = [a.get_text(strip=True) for td in td_tags for a in td.find_all('a')]

    links = standings_table.find_all('a')
    links = [l.get("href") for l in links]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    
    team_ids = [re.search(r'/squads/([^/]+)/', t_url).group(1) for t_url in team_urls if re.search(r'/squads/([^/]+)/', t_url)]
    team_name_links = [t_url.split("/")[-1].replace("-Stats", "") for t_url in team_urls]

        

    return team_urls, all_team_names, team_ids, team_name_links

In [4]:
def get_team_data(data, counter, team_name, fteam, opponent):

    if counter == 0:
        ScoresFixtures = pd.read_html(data.text, match="Scores & Fixtures")[0]
        ScoresFixtures.drop(['Match Report', 'Notes'], axis=1, inplace=True)
        fteam.append(ScoresFixtures)

    elif counter == 1:
        try:
            Shooting = pd.read_html(data.text, match="Shooting", header=1)
            # Shooting[0] = Shooting[0].rename(columns=lambda x: f'{team_name}_' + x)
            Shooting[1] = Shooting[1].rename(columns=lambda x: f'O_' + x)
            fteam.append(Shooting[0].iloc[:-1, 11:-1])
            opponent.append(Shooting[1].iloc[:-1, 11:-1])


        except Exception as e:
            print('The problem is ', e)
    elif counter == 2:
        try:
            Goalkeeping = pd.read_html(data.text, match="Goalkeeping", header=1)
            # Goalkeeping[0] = Goalkeeping[0].rename(columns=lambda x: f'{team_name}_' + x)
            Goalkeeping[1] = Goalkeeping[1].rename(columns=lambda x: f'O_' + x)
            
            Goalkeeping[0].drop('GA.1', axis=1, inplace=True)
            Goalkeeping[1].drop('O_GA.1', axis=1, inplace=True)
            
            fteam.append(Goalkeeping[0].iloc[:-1, 10:-1])
            opponent.append(Goalkeeping[1].iloc[:-1, 10:-1])
            

        except Exception as e:
            print('The problem is ', e)
    elif counter == 3:
        try:
            Passing = pd.read_html(data.text, match="Passing", header=1)
            # Passing[0] = Passing[0].rename(columns=lambda x: f'{team_name}_' + x)
            Passing[1] = Passing[1].rename(columns=lambda x: f'O_' + x)
            
            fteam.append(Passing[0].iloc[:-1, 10:-1])
            opponent.append(Passing[1].iloc[:-1, 10:-1])
            

        except Exception as e:
            print('The problem is ', e)
    elif counter == 4:
        try:
            PassTypes = pd.read_html(data.text, match="Pass Types", header=1)
            # PassTypes[0] = PassTypes[0].rename(columns=lambda x: f'{team_name}_' + x)
            PassTypes[1] = PassTypes[1].rename(columns=lambda x: f'O_' + x)
            
            fteam.append(PassTypes[0].iloc[:-1, 10:-1])
            opponent.append(PassTypes[1].iloc[:-1, 10:-1])
            

        except Exception as e:
            print('The problem is ', e)
            
    elif counter == 5:
        try:
            GoalShotCreation = pd.read_html(data.text, match="Goal and Shot Creation", header=1)
            # GoalShotCreation[0] = GoalShotCreation[0].rename(columns=lambda x: f'{team_name}_' + x)
            GoalShotCreation[1] = GoalShotCreation[1].rename(columns=lambda x: f'O_' + x)
            
            fteam.append(GoalShotCreation[0].iloc[:-1, 10:-1])
            opponent.append(GoalShotCreation[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
            
    elif counter == 6:
        try:
            DefensiveActions = pd.read_html(data.text, match="Defensive Actions", header=1)
            # DefensiveActions[0] = DefensiveActions[0].rename(columns=lambda x: f'{team_name}_' + x)
            DefensiveActions[1] = DefensiveActions[1].rename(columns=lambda x: f'O_' + x)            
                        
            fteam.append(DefensiveActions[0].iloc[:-1, 10:-1])
            opponent.append(DefensiveActions[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
            
    elif counter == 7:
        try:
            Possession = pd.read_html(data.text, match="Possession", header=1)
            # Possession[0] = Possession[0].rename(columns=lambda x: f'{team_name}_' + x)
            Possession[1] = Possession[1].rename(columns=lambda x: f'O_' + x)            
                                    
            fteam.append(Possession[0].iloc[:-1, 10:-1])
            opponent.append(Possession[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
            
    elif counter == 8:
        try:
            MiscellaneousStats = pd.read_html(data.text, match="Miscellaneous Stats", header=1)
            # MiscellaneousStats[0] = MiscellaneousStats[0].rename(columns=lambda x: f'{team_name}_' + x)
            MiscellaneousStats[1] = MiscellaneousStats[1].rename(columns=lambda x: f'O_' + x)            
                                                
            fteam.append(MiscellaneousStats[0].iloc[:-1, 10:-1])
            opponent.append(MiscellaneousStats[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    return fteam, opponent


In [5]:
session.cache.clear()

In [6]:
def league_proccess(league_link):
    base_data = session.get(league_link)
    League_soup = BeautifulSoup(base_data.text, features="lxml")

    # seasons_name = [
    #     "2022-2023",
    #     "2021-2022",
    #     "2020-2021",
    #     "2019-2020",
    #     "2018-2019",
    #     "2017-2018"
    #                 ]
    seasons_name = [
        # "2021",
        # "2022",
        "2023"]
    team_urls, all_team_names, team_id, team_name_l = get_team_links(base_data, League_soup)
    
    return seasons_name, team_urls, all_team_names, team_id, team_name_l

In [7]:
def team_stats(team_link, all_team_names, index, team_id, team_name_l):
    print(team_name_l[index])
    data = session.get(team_link)
    print(data.from_cache)
    if data.from_cache == False:
        time.sleep(60)
    soup = BeautifulSoup(data.text, features="lxml")
    # tables_urls = get_tables_links(data, soup)
    # tables_urls = [f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/schedule/{team_name_l[index]}-Scores-and-Fixtures-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/shooting/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/keeper/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/passing/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/passing_types/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/gca/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/defense/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/possession/{team_name_l[index]}-Match-Logs-All-Competitions',
    #                 f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/misc/{team_name_l[index]}-Match-Logs-All-Competitions']
    
    tables_urls = [f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/schedule/{team_name_l[index]}-Scores-and-Fixtures-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/shooting/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/keeper/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing_types/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/gca/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/defense/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/possession/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/misc/{team_name_l[index]}-Match-Logs-All-Competitions']
    
    TeamStatistics = pd.DataFrame()
    counter = 0

    fteam = []
    opponent = []
    for url in tables_urls:
        res = session.get(url)
        fteam, opponent = get_team_data(res, counter, all_team_names[index], fteam, opponent)
        counter += 1
        time.sleep(20)

    fteamdf = pd.concat(fteam, axis=1)
    opponentdf = pd.concat(opponent, axis=1)
    TeamStatistics = pd.concat([fteamdf, opponentdf], axis=1)
    TeamStatistics.insert(9, "Team", all_team_names[index])
    return TeamStatistics

In [8]:
def get_next_match(df):
    null_counts = df.isnull().sum(axis=1)

    first_row_to_drop = null_counts[null_counts > 295].index[0]
    next_match = df.drop(df.index[first_row_to_drop + 1:])
    return next_match.iloc[-1:, :]

In [ ]:
FutureMatche = pd.DataFrame()
for l_name, link in standings_urls.items():
    lastdata = []
    # NextMatches = []
    b_data = session.get(link)
    L_soup = BeautifulSoup(b_data.text, features="lxml")
    team_urls, all_team_names, team_id, team_name_l = get_team_links(b_data, L_soup)
    
    for index, team_link in enumerate(team_urls):
        # if l_name != 'PremierLeague':
        data = session.get(team_link)
        soup = BeautifulSoup(data.text, features="lxml")

        TeamStats = team_stats(team_link, all_team_names, index, team_id, team_name_l)
        
        lastdata.append(TeamStats)

        # NextMatches.append(get_next_match(TeamStats))
        
        print('*' * 100)
        time.sleep(100)
    newdfs = pd.concat(lastdata, axis=0)
    newdfs.to_csv(f"Datasets/{l_name}_2023-2024.csv", mode='w', index=False, encoding="utf-8")
    # if l_name in ['PremierLeague', 'Laliga', 'Bundesliga', 'SerieA', 'League1']:
    #     future_matches = pd.concat(NextMatches, axis=0)
    #     future_matches.to_csv(f"Datasets/LastSeasons/future_matches.csv", mode='w', index=False, encoding="utf-8")
    #     FutureMatche = pd.concat([FutureMatche, future_matches], axis=0)
    #     FutureMatche.to_csv(f"Datasets/LastSeasons/FutureMatche.csv", mode='a', index=False, encoding="utf-8")
        

In [ ]:
data_1 = pd.read_csv('Datasets/2017-2018.csv')
data_2 = pd.read_csv('Datasets/2018-2019.csv')
data_3 = pd.read_csv('Datasets/2019-2020.csv')
data_4 = pd.read_csv('Datasets/2020-2021.csv')
data_5 = pd.read_csv('Datasets/2021-2022.csv')
data_6 = pd.read_csv('Datasets/2022-2023.csv')
data_7 = pd.read_csv('Datasets/Women.csv')
data_8 = pd.read_csv('Datasets/SerieABrazil2021.csv')
data_9 = pd.read_csv('Datasets/SerieABrazil2022.csv')
data_10 = pd.read_csv('Datasets/SerieABrazil2023.csv')
data_11 = pd.read_csv('Datasets/PrimeraDivision2021.csv')
data_12 = pd.read_csv('Datasets/PrimeraDivision2022.csv')
data_13 = pd.read_csv('Datasets/PrimeraDivision2023.csv')

data_14 = pd.read_csv('Datasets/SerieA_2023-2024.csv')
data_15 = pd.read_csv('Datasets/PremierLeague_2023-2024.csv')
data_16 = pd.read_csv('Datasets/League1_2023-2024.csv')
data_17 = pd.read_csv('Datasets/Laliga_2023-2024.csv')
data_18 = pd.read_csv('Datasets/Eredivisie_2023-2024.csv')
data_19 = pd.read_csv('Datasets/Bundesliga_2023-2024.csv')
data_20 = pd.read_csv('Datasets/BelgianProLeague_2023-2024.csv')
data_21 = pd.read_csv('Datasets/PrimeiraLiga_2023-2024.csv')
data_22 = pd.read_csv('Datasets/PremierLeagueWomen_2023-2024.csv')

all_dfs = [data_1, data_2, data_3, data_4, data_5, data_6, data_7, data_8, data_9, data_10,
           data_11, data_12, data_13, data_14, data_15, data_16, data_17, data_18, data_19, data_20, data_21, data_22]


complete_dataset = pd.concat(all_dfs, axis=0)
complete_dataset.to_csv('Datasets/complete_dataset.csv', mode='w', index=False, encoding="utf-8")

⛔ Don't run this block anymore ⛔ This is for Premier League, Laliga, Bundesliga, Serie A and League 1

In [ ]:
seasons = [
    # "2022-2023/2022-2023-",
    # "2021-2022/2021-2022-",
    # "2020-2021/2020-2021-",
    # "2019-2020/2019-2020-",
    # "2018-2019/2018-2019-",
    "2017-2018/2017-2018-"]
# dfs = []
for season_index, season in enumerate(seasons):
     each_season = []

     league_urls = {
        # 'PremierLeague' : f'https://fbref.com/en/comps/9/{season}Premier-League-Stats',
        # 'Laliga' : f'https://fbref.com/en/comps/12/{season}La-Liga-Stats',
        # 'Bundesliga' : f'https://fbref.com/en/comps/20/{season}Bundesliga-Stats',
        # 'SerieA' : f'https://fbref.com/en/comps/11/{season}Serie-A-Stats',
        # 'League1' : f'https://fbref.com/en/comps/13/{season}ligue-1-Stats',
        # 'BelgianProLeague': f'https://fbref.com/en/comps/37/{season}Belgian-Pro-League-Stats',
        # 'Eredivisie': f'https://fbref.com/en/comps/23/{season}Eredivisie-Stats',
        'PrimeiraLiga': f'https://fbref.com/en/comps/32/{season}Primeira-Liga-Stats',
    }
     print(season)
     for league_name, league_link in league_urls.items():
        each_league = []
        seasons_name, team_urls, all_team_names, team_id, team_name_l = league_proccess(league_link)
        
        if season != "2017-2018/2017-2018-" and (league_name != 'Eredivisie' or league_name != 'PrimeiraLiga'):

            for index, team_link in enumerate(team_urls):

                TeamStatistics = team_stats(team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l) 
                # TeamStatistics.to_csv(f"Datasets/LastSeasons/{all_team_names[index]}_{seasons_name[season_index]}.csv", mode='w', index=False, encoding="utf-8")
                # dfs.append(TeamStatistics)
                each_season.append(TeamStatistics)
                each_league.append(TeamStatistics)
                print(TeamStatistics.shape)
                print('*' * 100)
                time.sleep(30)
            leagues = pd.concat(each_league, axis=0)
            leagues.to_csv(f"Datasets/{league_name}_{seasons_name[season_index]}.csv", mode='w', index=False, encoding="utf-8")
     seasonsDfs = pd.concat(each_season, axis=0, ignore_index=True)
     seasonsDfs.to_csv(f"Datasets/{seasons_name[season_index]}.csv", mode='w', index=False, encoding="utf-8")


⛔ The women league doesn't have season 2017-2018/2017-2018

In [ ]:
seasons = [
   # "2023-2024/2023/2024",
   # "2022-2023/2022-2023-",
   "2021-2022/2021-2022-",
   "2020-2021/2020-2021-",
   "2019-2020/2019-2020-",
   "2018-2019/2018-2019-",
   # "2017-2018/2017-2018-"
           ]


for season_index, season in enumerate(seasons):
   dfs = []

   league_urls = {
        'PremierLeagueWomen' : f'https://fbref.com/en/comps/189/{season}Womens-Super-League-Stats',
        # 'PremierLeague' : f'https://fbref.com/en/comps/9/{season}Premier-League-Stats',
        # 'Laliga' : f'https://fbref.com/en/comps/12/{season}La-Liga-Stats',
        # 'Bundesliga' : f'https://fbref.com/en/comps/20/{season}Bundesliga-Stats',
        # 'SerieA' : f'https://fbref.com/en/comps/11/{season}Serie-A-Stats',
        # 'League1' : f'https://fbref.com/en/comps/13/{season}ligue-1-Stats',
        # 'BelgianProLeague': f'https://fbref.com/en/comps/37/{season}Belgian-Pro-League-Stats',
        # 'Eredivisie': f'https://fbref.com/en/comps/23/{season}Eredivisie-Stats',
        # 'PrimeiraLiga': f'https://fbref.com/en/comps/32/{season}Primeira-Liga-Stats',
    }
   print(season)
   for league_name, league_link in league_urls.items():
      seasons_name, team_urls, all_team_names, team_id, team_name_l = league_proccess(league_link)

      for index, team_link in enumerate(team_urls):

         TeamStatistics = team_stats(team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l) 

         dfs.append(TeamStatistics)
   fdfs = pd.concat(dfs, axis=0)
   fdfs.to_csv(f"Datasets/_{season[season_index]}_.csv", mode='w', index=False, encoding="utf-8")

⛔ The south American League doesn't have the season before 2021

In [ ]:
seasons = [
    # "2021/2021",
    # "2022/2022",
    "2023/2023"]
for season_index, season in enumerate(seasons):
     dds = []

     league_urls = {
        'PrimeraDivision' : f'https://fbref.com/en/comps/21/{season}-Primera-Division-Stats',
        'SerieABrazil' : f'https://fbref.com/en/comps/24/{season}-Serie-A-Stats',
    }
     print(season)
     for league_name, league_link in league_urls.items():
        e_league = []
        seasons_name, team_urls, all_team_names, team_id, team_name_l = league_proccess(league_link)

        for index, team_link in enumerate(team_urls):
            TeamStatistics = team_stats(team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l) 
            print(TeamStatistics.shape)
            dds.append(TeamStatistics)
            e_league.append(TeamStatistics)
            print('*' * 100)
            time.sleep(200)
        eax_league = pd.concat(e_league, axis=0)
        if league_name == 'PrimeraDivision':
            eax_league.to_csv(f"Datasets/PrimeraDivision2023.csv", mode='w', index=False, encoding="utf-8")
        elif league_name == 'SerieABrazil':
            eax_league.to_csv(f"Datasets/SerieABrazil2023.csv", mode='w', index=False, encoding="utf-8")

    #  f_dds = pd.concat(dds, axis=0)
    #  TeamStatistics.to_csv(f"Datasets/_{seasons[season_index]}_.csv", mode='w', index=False, encoding="utf-8")